# Session 3 — DICE — Notebook 1 (Basics)

This notebook is a **minimal teaching scaffold** to help you (and students) simulate the DICE-style model step by step and produce a few baseline plots.  
Scope (kept intentionally small):
- Load the provided `DICE.py` (parameters, state-update, light planner scaffolding).
- Build the time grid and initial state matrix.
- Pick **one simple control path** (abatement `μ_t`) and simulate.
- Plot **one endogenous path** (e.g., temperature `T_AT`) and **one exogenous path** (e.g., non-CO₂ forcing `F_EX`).

> Advanced/optimization content is **not** the focus here; see Notebook 2 for the control problem.


## Session 3 — Objectives and prerequisites

**Learning objectives**
- Identify the economic and climate blocks of the original `DICE.py` model.
- Run a controlled BAU-versus-transition comparison.
- Build and compare SSP parameterizations, then construct one mitigation ramp.

**Prerequisites.** Slides for Session 3; Sessions 1–2; comfort with NumPy matrices and model indices stored in `Params`.


## Technical setup

Run the notebook from this session directory. The original course module and data files are stored next to it.

**Convention.** At the initial date, control variables are `NaN`: the initial state is given, but the control is chosen only for subsequent transitions.


In [ ]:
# Install dependencies in the current notebook kernel before running imports.
%pip install -r requirements.txt


## 0) Setup


In [ ]:
# Standard imports
import sys, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


## 1) Import the DICE module & inspect parameters


In [ ]:
from DICE import *

# Create a default parameter object
p = Params()
print(p)
print("\nNumber of periods nT =", p.nT, "with step Δ =", p.Delta, "years")
print("Time runs from", p.t0, "to", p.tT)


## 2) Build initial state matrix
### Time Structure and Variables in the DICE Model

Before setting up the numerical solution, let us clarify the time indexing and the role of variables:

- We work in **discrete time** with finite horizon $T$.
- Periods are indexed by $i=0,1,2,\dots,I$ with time step $\Delta$ years.
- Calendar time is $t(i) = t_0 + i\Delta$.

Thus, the time grid is:
$$
t = \{t_0, t_0 + \Delta, t_0 + 2\Delta, \dots, T\}.
$$

- Variables can be written as $Y_t$ (calendar time) or $Y(i)$ (discrete index).

### Setting exogenous states
Exogenous state variables are given by $x_t \in \mathbb{R}^{N_x}$: productivity $A_t$, carbon intensity $\sigma_t$, radiative forcing $F_t^{EX}$, land-use emissions $E_t^{land}$, population $L_t$, etc.

They follow an exogenous process:
  $$
  x_t = g(x_{t-1}), \quad g: \mathbb{R}^{N_x} \to \mathbb{R}^{N_x}.
  $$

They are next stored in to $y$ matrix, other variabels are yet left NaN.


In [ ]:
# Initialize the simulation matrix with exogenous drivers and initial states
sim = init_states(p)

# Convert to a tidy DataFrame for inspection (optional)
df0 = mat_to_df(sim, p)
df0.head()

## 3) Choose a simple control path, then simulate

The vector of  **Endogenous states/flows** $y_t \in \mathbb{R}^{N_y}$: carbon stocks $(M_t^{AT}, M_t^{UP}, M_t^{LO})$, temperatures $(T_t^{AT}, T_t^{OC})$, output $Y_t$, capital $K_t$, etc. They evolve according to
  $$
  y_t = f(y_{t-1}, x_{t-1}, z_t), \quad f: \mathbb{R}^{N_y}\times \mathbb{R}^{N_x}\times \mathbb{R}^2 \to \mathbb{R}^{N_y}.
  $$

By setting **decision variables (controls)** $z_t = [\mu_t, s_t]' \in \mathbb{R}^2$: abatement rate and saving rate, we are able to numerically solve the path of the economy.

### Numerical Resolution (Open Loop)
Under Assumptions (1) and (2), decisions are exogenous and the model can be solved recursively:
1. Update exogenous states: $x_t = g(x_{t-1})$.
2. Compute endogenous states: $y_t = f(y_{t-1}, x_{t-1}, z_t)$.

This provides the full trajectory $(x_t, y_t, z_t)$ for $t = t_0, \dots, T$.

### In this example
For this **basics** notebook we avoid optimization. We simply **feed exogenous controls**, then call the **state updater**:
- Fix saving rate `s_t` to a constant (e.g., 20%).
- Set a **ramp** for the abatement rate `μ_t` to 3%  (purely illustrative).


In [ ]:
# Time indices to update (DICE arrays start at t=1 for transitions)
timevec = range(1, p.nT)

# 3.a) Controls: saving and abatement (illustrative — not optimal)
sim[1:, p.i_s]  = 0.20  # constant saving rate at 20%
# Start abatement at 3%
sim[1:, p.i_mu] =0.03

# 3.b) Propagate the system forward using the update function
sim = update_path(sim, timevec, p)

# Show a couple lines to confirm variables updated (e.g., Y, C, T_AT, F_EX)
cols_to_show = ['time','s','mu','Y','C','T_AT','F_EX','E_land']
pd.DataFrame(sim, columns=p.col)[cols_to_show].head(8)


## 4) Plot: one endogenous path and one exogenous driver


In [ ]:
years = sim[:, p.i_time]

# Endogenous example: atmospheric temperature T_AT
plt.figure()
plt.plot(years, sim[:, p.i_T_AT], linewidth=2)
plt.xlabel("Year"); plt.ylabel("Atmospheric temperature (°C)")
plt.title("Endogenous path: $T_{AT}$")
plt.grid(True)
plt.show()

# Exogenous example: non-CO₂ forcing F_EX
plt.figure()
plt.plot(years, sim[:, p.i_F_EX], linewidth=2)
plt.xlabel("Year"); plt.ylabel("Non-CO$_2$ forcing (W/m$^2$)")
plt.title("Exogenous driver: $F_{EX}$")
plt.grid(True)
plt.show()


## 5) Optional: quick look at Output and Consumption


In [ ]:
plt.figure()
plt.plot(years, sim[:, p.i_Q], linewidth=2, label="Output Q")
plt.plot(years, sim[:, p.i_C], linewidth=2, label="Consumption C")
plt.xlabel("Year"); plt.ylabel("Level (model units)")
plt.title("Output and Consumption (illustrative controls)")
plt.grid(True); plt.legend()
plt.show()


## 6) Question: Compare a laissez-faire scenario (no abatement $\mu=0$) against a green transition scenario (abatement starting at 0 in 2020 and reaching 100\% linearly in 2050). Report in a graph.


In [ ]:
# Your code here. Here is an help:
sim_BAU = sim.copy()
sim_BAU[1:, p.i_s]  = 0.20  # constant saving rate at 20%
sim_BAU[1:, p.i_mu] = 0

---

*Original notebook continuation: `DICE_Notebook1-2_generation_scenarios.ipynb`.*


# DICE — Notebook 2 (Exercises): Scenarios & Forcing


**Learning objectives (1h30min)**
0. Load a minimal DICE code and identify its **blocks**: parameters, initial states, economic equations, climate equations.  
1. SSP Scenarios: generate meaningful SSP scenarios by adjusting the calibration of exogenous variables.
2. Introduce some forcing objectives, by adjusting

> This notebook uses the provided `DICE.py` module (teaching version).


# 0) Load DICE and run baseline


In [ ]:

# 1) Load DICE and baseline run
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import DICE 

p = Params()
path = init_states(p)
path[1:, p.i_mu] = 0.03          # baseline abatement path
path = update_path(path, range(1, p.nT), p)
df_base = mat_to_df(path, p)
df_base.head()


## 1) SSP Scenarios

In what follows, we want to simulate two extreme narratives, the **SSP1** and **SSP5** storylines, as well as the **baseline (SSP2)**.  

- **SSP1** – *Sustainability: Taking the Green Road (Low challenges to mitigation and adaptation)*  
  *The world shifts gradually toward a more sustainable path, emphasizing inclusive development and respect for environmental boundaries. Educational and health investments accelerate the demographic transition, inequality is reduced, and consumption patterns shift toward low material growth and lower resource and energy intensity.*

- **SSP5** – *Fossil-fueled Development: Taking the Highway (High challenges to mitigation, low challenges to adaptation)*  
  *This world emphasizes competitive markets, innovation, and human capital development. Rapid economic growth is fueled by fossil resources and energy-intensive lifestyles. Local environmental problems are managed, but global commons are stressed, with high emissions and reliance on future technological fixes or geo-engineering.*

We thus consider three scenarios: the **optimistic** (SSP1), the **baseline** (SSP2, current calibration), and the **pessimistic** (SSP5).  
The table below summarizes the proposed core calibration changes:

| Parameter (description)                | Symbol   | SSP1 (Optimistic)                         | SSP2 (Baseline — DICE) | SSP5 (Pessimistic)                        |
|----------------------------------------|----------|--------------------------------------------|-------------------------|-------------------------------------------|
| Emissions intensity (initial)          | $\sigma_0$ | 25% lower than baseline                     | calibrated value        | 15% higher than baseline                   |
| Decline rate of emissions intensity    | $g_{\sigma}$ | 25% faster decline                         | calibrated value        | 50% slower decline                        |
| Long-run population (asymptote)        | $L_{\infty}$ | 9 billion                                  | calibrated value        | 16 billion                                |
| Population growth parameter            | $l_g$      | 20% lower                           | calibrated value        | 30% higher                        |
| Total factor productivity (TFP) level  | $A_0$      | 5% higher than baseline                    | calibrated value        | 3% lower than baseline                    |
| TFP growth rate                        | $g_A$      | 15% faster than baseline                   | calibrated value        | 20% slower than baseline                  |

>  **Instructions**: For each scenario, initialize parameters accordingly, run the model, and compare trajectories for output, emissions, forcing, and temperatures.


#### 1-A) Build the calibration and check it works in new p vector.
Use the table to fill each scenarios and have the right narrative.


In [ ]:
# Your code for question 1-A should be here

def make_params_variant(**kwargs):
    q = Params()
    for k, v in kwargs.items():
        setattr(q, k, v)
    return q

p_base = Params()

# Worked example: SSP1 (optimistic)
p_opt = make_params_variant(
    sigma0=p_base.sigma0 * 0.75,  # 25% lower initial emissions intensity
    gsig=p_base.gsig * 1.25,      # 25% faster decline
    Linf=9000,                    # long-run population: 9 billion
    lg=p_base.lg * 0.80,          # 20% lower population growth parameter
    A0=p_base.A0 * 1.05,          # 5% higher initial TFP
    gA=p_base.gA * 1.15,          # 15% faster TFP growth
)

# Student task: implement SSP5 (pessimistic) using the calibration table above.
# p_pes = make_params_variant(sigma0=..., gsig=..., Linf=..., lg=..., A0=..., gA=...)


#### 1-B) Provide some graph comparing Emissions, GDP, damages (define them manually) and temperatures.


In [ ]:
# Your code for question 1-B should be here

### 1-B) Report the graph and compare them
sim_bas = init_states(p_base)
#sim_opt = init_states(p_opt)
#sim_pes = init_states(p_pes)


sim_bas[1:, p_base.i_s]  = 0.20  # constant saving rate at 20%
sim_bas[1:, p_base.i_mu] = 0.03
timevec = range(1, p.nT)
sim_bas = update_path(sim_bas, timevec, p_base)


#### 1-C) According to all the figures, do you think that SSPs narrative are the main drivers of climate change? What is the temperatures in best versus worst case of climate change?


Write your answer here.


## 2) Mitigation schedules: ramp abatement to reach (near) net-zero by 2100 / 2080 / 2060

Up to now, our scenarios were purely **exogenous**: we varied productivity, population, and emissions intensity to mimic different SSP narratives.  
We now enrich the analysis by introducing a **second decision variable**: the **abatement rate** $\mu_t$, which measures the fraction of industrial emissions avoided through mitigation policies and technologies.  

- In the **baseline** exercises, $\mu_t$ was fixed at a constant, low level (e.g. 3%).  
- In this exercise, we explicitly feed an **abatement trajectory** $\{\mu_t\}$ into the model.  
- Conceptually, pushing $\mu_t \to 1$ means that almost all industrial CO₂ emissions are eliminated. Note, however, that **land-use emissions** $E_{\text{land},t}$ remain exogenous and may stay positive for some time.

This mirrors the way scenarios are constructed in **CMIP6 (Coupled Model Intercomparison Project Phase 6)**, where each SSP storyline can be combined with different **climate policy assumptions**:  
- A **high-ambition mitigation pathway** ramps abatement quickly, reaching **net-zero emissions** by mid-century (e.g. 2060).  
- A **delayed mitigation pathway** reaches net-zero much later (e.g. 2100).  
- The resulting **forcing levels by 2100** (e.g. 2.6, 4.5, 8.5 W/m²) define the well-known **SSP1-2.6**, **SSP2-4.5**, and **SSP5-8.5** scenarios.  

3. Interpret the results in light of the IPCC’s SSP scenario framework: *early and ambitious decarbonization (SSP1-2.6) vs. delayed action with high warming (SSP5-8.5).*


#### 2-a) Construct linear ramps for $\mu_t$ that reach **net zero** by 2100, 2080, and 2060. 
Recall that in this model, we only have two decision variables in here. Saving remains unchanged $\{s_t\}_{t=t_{0}}^{T}=0.2$ while mitigation path must be adjusted: $\{\mu_t\}_{t=t_{0}}^{T}$. It should gradually increase to 1 to reach net zero at some specific date.
Recall that here, our simulations, including decision variables, are all stacked into matrix $w = [y,x,z]$. In what follows, we modify directly $w$.


In [ ]:
# Worked example: a linear abatement ramp reaching 100% in 2100
years = sim_bas[:, p_base.i_time]
mu_initial = 0.03
target_year = 2100

ramp2100 = np.clip(
    mu_initial + (1 - mu_initial) * (years - years[0]) / (target_year - years[0]),
    mu_initial,
    1.0,
)

plt.figure(figsize=(7, 4))
plt.plot(years, ramp2100, linewidth=2, label="Net zero in 2100")
plt.xlabel("Year")
plt.ylabel(r"Abatement rate $\mu_t$")
plt.ylim(0, 1.05)
plt.grid(True)
plt.legend()
plt.show()

# Student task: construct analogous ramps reaching 100% in 2080 and 2060.
# ramp2080 = ...
# ramp2060 = ...


#### 2-b) Simulate the model under each ramp and compare the paths for **forcing $F_t$**, **temperatures $T_{AT,t}$**, and **emissions $E_t$**.
Recall that once the matrix of decision $z$ of size $I\times 2$ has been modified (as part of matri $w$), we can once again solve the new system with alternative climate policy:
  $$
  y_t = f_p (y_{t-1}, x_{t-1}, z_t), \quad f: \mathbb{R}^{N_y}\times \mathbb{R}^{N_x}\times \mathbb{R}^2 \to \mathbb{R}^{N_y}.
  $$

  How effective is climate policy in curbing emissions?


In [ ]:
# Worked example: simulate the model with the 2100 ramp defined above
sim2100 = sim_bas.copy()
sim2100[1:, p_base.i_s] = 0.20
sim2100[1:, p_base.i_mu] = ramp2100[1:]
sim2100 = update_path(sim2100, range(1, p_base.nT), p_base)

# Check the resulting forcing, temperature, and emissions paths
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(years, sim2100[:, p_base.i_F], linewidth=2, label="Net zero in 2100")
axes[0].set_xlabel("Year")
axes[0].set_ylabel(r"W/m$^2$")
axes[0].set_title("Forcing")
axes[0].grid(True)
axes[0].legend()

axes[1].plot(years, sim2100[:, p_base.i_T_AT], linewidth=2, label="Net zero in 2100")
axes[1].set_xlabel("Year")
axes[1].set_ylabel(r"$^{\circ}$C")
axes[1].set_title("Atmospheric temperature")
axes[1].grid(True)
axes[1].legend()

axes[2].plot(years, sim2100[:, p_base.i_E], linewidth=2, label="Net zero in 2100")
axes[2].set_xlabel("Year")
axes[2].set_ylabel(r"GtCO$_2$")
axes[2].set_title("Emissions")
axes[2].grid(True)
axes[2].legend()

plt.tight_layout()
plt.show()

# Student task: repeat the same procedure with ramp2080 and ramp2060,
# then add the corresponding trajectories to the comparison graphs.
# sim2080 = ...
# sim2060 = ...


Write your answer here.


### 2-c) What is the economic cost of climate mitigation? 
Provide a 2 plots, one showing the consumption gap defined as $100\times (c_t^{alt}/c_t^{base}-1)$ and in a second figure the temperature gap  $(T_t^{alt}-T_t^{base})$. $T_t^{alt}$ denotes the temperature path in alternative regimes (e.g. optimistic or pessimistic) while *base* is baseline.  
Comment the result your find about the sacrifice that must be done by society in terms of consumption loss during the transition. What drives this sacrifice?


In [ ]:
# Your code here.

Write your answer here.


## 3) Providing core scenarios mixing SSPs $\times$ Forcing as in IPCC

The table below shows how **SSP storylines** combine with **radiative forcing targets** (W/m² by 2100).  
For example, *SSP1-2.6* corresponds to the **SSP1 narrative** combined with a **2.6 W/m² forcing pathway**.

|      | **1.9** | **2.6** | **4.5** | **7.0** | **8.5** |
|------|---------|---------|---------|---------|---------|
| **SSP1** | SSP1-1.9 | SSP1-2.6 | –       | –       | –       |
| **SSP2** | –       | SSP2-2.6 | SSP2-4.5 | SSP2-7.0 | –       |
| **SSP3** | –       | –       | SSP3-4.5 | SSP3-7.0 | SSP3-8.5 |
| **SSP4** | –       | SSP4-2.6 | SSP4-4.5 | SSP4-7.0 | –       |
| **SSP5** | –       | –       | SSP5-4.5 | –       | SSP5-8.5 |

### Approximate Mapping: Forcing → Global Warming by 2100

Radiative forcing pathways translate into different **global mean temperature increases** relative to preindustrial levels (1850–1900).  
Values below are **approximate central estimates** (IPCC AR6, median climate sensitivity assumptions):

- **1.9 W/m² (SSP1-1.9)** → ~ **1.5 °C** stabilization (consistent with Paris Agreement “1.5 °C” target).  
- **2.6 W/m² (SSP1/2/4-2.6)** → ~ **2.0 °C** warming by 2100 (low-forcing pathway).  
- **4.5 W/m² (SSP2/3/4/5-4.5)** → ~ **2.5–3.0 °C** warming by 2100 (intermediate pathway).  
- **7.0 W/m² (SSP2/3/4-7.0)** → ~ **3.5–4.0 °C** warming by 2100 (high-forcing pathway).  
- **8.5 W/m² (SSP3/5-8.5)** → ~ **4.5 °C or more** warming by 2100 (very high emissions, “worst-case” reference).

---

 **Interpretation**:  
- Lower forcing pathways (SSP1-1.9, SSP1/2-2.6) require **early, ambitious mitigation** and typically correspond to net-zero CO₂ by mid-century.  
- Higher forcing pathways (SSP3-7.0, SSP5-8.5) assume **limited mitigation**, continued fossil fuel use, and delayed or absent net-zero transitions.  
- Intermediate cases (SSP2-4.5) represent **current policy trajectories** or delayed mitigation efforts.


#### 3-A) Some core scenarii outside the diagonal are not evaluated (e.g. SSP1-8.5). Why?


Write your answer here.


#### 3-B) Try to compute the SSP1-1.9, which corresponds to net zero by 2060 and optimistic scenario. Is SSP1-1.9 out of reach?


In [ ]:
# Your code here.

Write your answer here.


#### 3-C) Is it possible to obtain a 4°C of warming in SSP1? Evaluate SSP1 under no mitigation policy with $\mu=0$ to mimic SSP1-8.5


In [ ]:
# Your code here

Write your answer here.


#### 3-D) Is it possible to obtain a 1.5°C in pessimistic environment? Evaluate SSP5 under mitigation policy with $\mu=1$ by 2060 to mimic SSP5-1.9


In [ ]:
# Your code here

Write your answer here.
